# 06: Range Trees

*Authors: Felix Espey, Kevin Buchin*

This notebook serves as supplementary learning material for the course **Geometric Algorithms**.
It showcases and explains implementations of algorithms presented in the corresponding lecture, and elaborates on some practical considerations concerning their use.
Furthermore, it offers interactive visualisations and animations.

## Table of Contents

1. Introduction
2. Range Searching in one Dimension
3. Range Searching in two Dimensions (Kd-Trees)
4. Range Trees
5. References

## 1. Introduction



## 2. Range Searching in one dimension

The goal of a one dimensional range search is to find all entries in a set (of numbers) which are within a given range. The algorithm presented in the lecture uses a binary search tree in which the inner nodes are used solely as splitting nodes and the data is stored in the leaves.
The search starts at the root of the tree and descents until a node is found which is within the range. This node is called the splitting node. Since all nodes in the left subtree are smaller than the splitting node, they are guaranteed to be smaller than the right border of the range, so a simple comparison with the left boundary of the range is enough. Similarly, all nodes in the right subtree are guaranteed to be bigger than the left border of the range, so a comparison with the right boundary is sufficient.

Since the drawings used in the notebook to not present good ways to input the left and right boundaries used for this section, they are instead defined as parameters at the start of each subsection. To change the range simply change the parameters above the drawing you are currently at.

The algorithms also include lines like
- tag_current_node(some integer)
- go_to_left_child
- go_to_parent
All of these lines are only needed for the drawing and are not part of the actual algorithm.

Lastly, the implementations below all consist of methods which take a tree Node as an argument. Usually this would be implemented as methods of the node class, but since the Node class is very large and features a lot of method not needed here, it is instead moved to the backend and the method here take a node as an argument

### 2.1 Finding the splitting node

As stated before the first step for completing a range search is to find the first node that is within the range. The method "find_splitting_node" below does exactly that, descending through the tree from the given node until a node within the range is found. An important note here is that the construction of the tree guarantees that each node has either 0 or 2 children, which simplifies some checks.
There are a total of five cases that need to be accounted for:

- The current node is a leaf and outside the range. Then no splitting node exists and the return value is None. In the drawing this node is marked red
- The current node is inside the range. Then the current node is returned. In the drawing this node is marked green
- The current node outside the range and is not a leaf. The algorithm then further descents the tree, going right if the current node is to the left of the range and going left if the current node is on the right. In the drawing these nodes are marked blue

In [1]:
from __future__ import annotations
from modules.data_structures.binary_trees import EST
from modules.visualisation import VisualisationTool, BinaryTreeInstance
from modules import Node, IntComparator, ComparisonResult, RangeSearchAnimator, RangeSearchMode

# -------- change boundaries here
LOWER_BOUND = 10
UPPER_BOUND = 50
# --------

# -------- visualization setup
bti = BinaryTreeInstance()
vis = VisualisationTool(400,400, bti)
# --------

int_comparator = IntComparator()

def find_splitting_node(node : Node[int, None], lower_bound : int, upper_bound : int, rta : RangeSearchAnimator) -> Node[int, None] | None:
    cr_left = int_comparator.compare(lower_bound, node.key)
    cr_right = int_comparator.compare(upper_bound, node.key)
    if cr_right is ComparisonResult.BEFORE:
        #range fully left of node
        if node.left is None:
            rta.tag_cur_node(3)
            return None
        else:
            rta.tag_cur_node(1)
            rta.go_to_left_child()
            return find_splitting_node(node.left, lower_bound, upper_bound, rta)
    elif cr_left is ComparisonResult.AFTER:
        #range fully right of node
        if node.right is None:
            rta.tag_cur_node(3)
            return None
        else:
            rta.tag_cur_node(1)
            rta.go_to_right_child()
            return find_splitting_node(node.right, lower_bound, upper_bound, rta)
    else:
        #node in range
        rta.tag_cur_node(2)
        rta.save_node() # needed to find node in range search
        return node

def run_find_splitting_node(est : EST[int]) -> RangeSearchAnimator:
    rta = RangeSearchAnimator(est)
    if LOWER_BOUND < UPPER_BOUND:
       find_splitting_node(est._root, LOWER_BOUND, UPPER_BOUND, rta)
    return rta

# -------- visualization
vis.register_algorithm("find splitting node", run_find_splitting_node, RangeSearchMode())
vis.display()

Once a splitting node is found, the next step is to report all nodes within the range from its left and right subtree. To achieve this we use to helper method, less_or_equal and greater_or_equal, which report all nodes in a given subtree that fulfill the boundary requirement.

In less_or_equal the current node is compared to the right boundary of the range. This gives the following cases:

- The node is left of the boundary and not a leaf. Then the all leaves in the left subtree are reported and the algorithm descents to the right child of the node
- The node is left of the boundary and a leaf. Then the current node is reported
- The node is right of the boundary and not a leaf. Then the algorithm descents to the left child.
- The node is right of the boundary and a leaf. Then the algorithm returns without reporting the node.

The greater_or_equal method is very similar, as it does the same but flips the decision and direction of descent. The cases are as follows:

- The node is right of the boundary and not a leaf. Then the all leaves in the right subtree are reported and the algorithm descents to the left child of the node
- The node is right of the boundary and a leaf. Then the current node is reported
- The node is left of the boundary and not a leaf. Then the algorithm descents to the right child.
- The node is left of the boundary and a leaf. Then the algorithm returns without reporting the node.

All visited nodes are marked as blue in the drawing. All reported nodes are marked in green and all leaves that where visited but not reported are marked in red.

It is important to note that the implementations of less_or_equal and greater_or_equal below start at the root instead of the splitting node. This lets their results overlap in many cases.

In [2]:
# -------- change boundaries here
LOWER_BOUND = 80
UPPER_BOUND = 200
# --------

def leaves(node : Node[int, None], rta:RangeSearchAnimator):
    if node.is_leaf():
        rta.tag_cur_node(2)
    else:
        rta.tag_cur_node(1)
        rta.go_to_left_child()
        leaves(node.left, rta)
        rta.go_to_parent()
        rta.go_to_right_child()
        leaves(node.right, rta)
        rta.go_to_parent()

def less_or_equal(node : Node[int, None], upper_bound : int, rta : RangeSearchAnimator):
    cr = int_comparator.compare(upper_bound, node.key)
    if cr is ComparisonResult.MATCH or cr is ComparisonResult.AFTER:
        #less than search term
        if not node.is_leaf():
            rta.tag_cur_node(1)
            rta.go_to_left_child()
            leaves(node.left, rta)
            rta.go_to_parent()
            rta.go_to_right_child()
            less_or_equal(node.right, upper_bound, rta)
            rta.go_to_parent()
        else:
            rta.tag_cur_node(2)
    else:
        #more than search term
        if not node.is_leaf():
            rta.tag_cur_node(1)
            rta.go_to_left_child()
            less_or_equal(node.left, upper_bound, rta)
            rta.go_to_parent()
        else:
            rta.tag_cur_node(3)

def greater_or_equal(node : Node[int, None], lower_bound : int, rta : RangeSearchAnimator):
    cr = int_comparator.compare(lower_bound, node.key)
    if cr is ComparisonResult.BEFORE or cr is ComparisonResult.MATCH:
        #more than search term
        if not node.is_leaf():
            rta.tag_cur_node(1)
            rta.go_to_left_child()
            greater_or_equal(node.left, lower_bound, rta)
            rta.go_to_parent()
            rta.go_to_right_child()
            leaves(node.right, rta)
            rta.go_to_parent()
        else:
            rta.tag_cur_node(2)
    else:
        #less than search term
        if not node.is_leaf():
            rta.tag_cur_node(1)
            rta.go_to_right_child()
            greater_or_equal(node.right, lower_bound, rta)
            rta.go_to_parent()
        else:
            rta.tag_cur_node(3)

def run_less_or_equal(est : EST[int]) -> RangeSearchAnimator:
    rta = RangeSearchAnimator(est)
    less_or_equal(est._root,UPPER_BOUND , rta)
    return rta

def run_greater_or_equal(est : EST[int]) -> RangeSearchAnimator:
    rta = RangeSearchAnimator(est)
    greater_or_equal(est._root,LOWER_BOUND , rta)
    return rta

vis.register_algorithm("report smaller than upper bound", run_less_or_equal, RangeSearchMode())
vis.register_algorithm("report bigger than lower bound", run_greater_or_equal, RangeSearchMode())
vis.display()

With the methods defined above the range search algorithm becomes pretty simple. First the splitting node for the given range is searched. If no splitting node is found the algorithm terminates with an empty list. If the splitting node is a leaf it is returned alone. If a left subtree exists, all nodes greater or equal to the left boundary of the range are added to the result. If a right subtree exists, all nodes lesser or equal t o the right boundary of the range are also added.

The drawing marks all visited nodes in blue, all nodes within the range in green and all nodes leaves that where visited but not reported in red.

In [3]:
# -------- change boundaries here
LOWER_BOUND = 80
UPPER_BOUND = 200
# --------

def range_search(node : Node[int, None], lower_bound: int, upper_bound : int, rta : RangeSearchAnimator) -> list[Node[int, None]]:
        splitting_node = find_splitting_node(node, lower_bound, upper_bound, rta)
        if splitting_node is None:
            return []
        if splitting_node.is_leaf():
            return [splitting_node]
        else:
            rta.load_node()
            rta.tag_cur_node(1)
            result = []
            if splitting_node.left is not None:
                rta.go_to_left_child()
                greater_or_equal(splitting_node.left, lower_bound, rta)
                rta.go_to_parent()
            if splitting_node.right is not None:
                rta.go_to_right_child()
                less_or_equal(splitting_node.right, upper_bound, rta)
                rta.go_to_parent()
            return result

def run_range_search(est : EST[int]) -> RangeSearchAnimator:
    rta = RangeSearchAnimator(est)
    if LOWER_BOUND < UPPER_BOUND:
        range_search(est._root, LOWER_BOUND, UPPER_BOUND, rta)
    return rta

vis.register_algorithm("report in range", run_range_search, RangeSearchMode())
vis.display()
vis._animation_checkbox.value = True
vis._random_button.click()
vis._algorithm_buttons[3].click()


## 3. Range Searching in two Dimensions (Kd-Trees)



In [4]:
from modules import Point, AnimationObject
from enum import Enum
from typing import Optional, Iterator
from modules import PointSetInstance

class KDNode:
    class Axis(Enum):
        X = 0
        Y = 1

    def __init__(self, point : Point, axis : Axis = Axis.X, left : Optional[KDNode]=None, right : Optional[KDNode]=None):
        self.point : Point = point
        self.axis : KDNode.Axis = axis        # 0 for x, 1 for y
        self.left : Optional[KDNode] = left
        self.right : Optional[KDNode] = right

class KDTree2D(AnimationObject):

    def __init__(self, points: list[Point] = None):
        super().__init__()
        self.root = None
        if points:
            self.root = self.build(points, depth = 0)

    def points(self) -> Iterator[Point]:
        if not self.root:
            return iter([])
        nodes = [self.root]
        points = []
        while len(nodes) > 0:
            cur = nodes.pop()
            points.append(cur.point)
            if cur.left:
                nodes.append(cur.left)
            if cur.right:
                nodes.append(cur.right)
        return iter(points)

    def build(self, points : list[Point], depth : int) -> Optional[KDNode]:
        if not points:
            return None
        axis = depth % 2
        if len(points) == 1:
            return KDNode(points[0], axis)
        if axis is KDNode.Axis.X:
            points.sort(key=lambda point : point.x)
        else:
            points.sort(key=lambda point : point.y)
        median = len(points) // 2
        left = self.build(points[:median], depth + 1)
        right = self.build(points[median + 1:], depth + 1)
        return KDNode(
            point=points[median],
            axis=axis,
            left=left,
            right=right,
        )

test = [Point(0, 1), Point(0,2), Point(1,0), Point(2,0)]

tree = KDTree2D(test)



## 4. Range Trees
TODO